scale: 10dBV/  
ref level: -50 dBV  
span: 1 kHz  
center: 400Hz  
window: hanning  
max points: 2M  
fft mode: normal  
search threshold: -160 dBV  
search excursion: 20 dB  
sort by: amplitude  
  
draadlengte (die beweegt): 31.5 +-0.1 cm  
metingen op: 8.9 +-0.1 cm  
1ste plek voor gewicht: 8 +-0.1 cm  
ruimte tussen plekken: 4 +-0.1 cm  
ruimte aan andere kant: 2 +- 0.1 cm  
gewicht: 1.0468 +-0.0001 kg  
gewicht 2: 1.0436 +-0.0001 kg

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
from scipy.signal import find_peaks, butter, filtfilt

In [ ]:
#IMPORTING ALL FILES

#this script imports all data but keeps file structure intact by putting it in a dictionary
#print(results.keys()) shows all folders of different distances
#print(results["metingen x cm"].keys()) shows all positions used with distance x
#print(results["metingen x cm"]["metingen y-z"].keys()) shows all measurements that were taken with distance x in position y-z
#here y is the position of the lighter weight and z is the position of the heavier one

#results["metingen 30 cm"]["metingen 4-4"]["fft_1"] calling on the data like this, using the same names as the file structure
# (except for the csv file names, they have been changed to "fft_x" and "raw_x" to be more clear)

#use this to loop over all files, and print info:

# for cm_name, positions in results.items():
#     for position, measurements in positions.items():
#         for measurement_name, data in measurements.items():
#             print(f"{cm_name} / {position} / {measurement_name}: shape={data.shape}")

base_dir = Path("metingen")

position_names = [f"0-{i}" for i in range(1, 5)]+[f"{i}-4" for i in range(1, 5)]

def numbered_names(prefix, count):
    return [prefix] + [f"{prefix} ({i})" for i in range(1, count)]

measurement_names = numbered_names("usr_wf_C1_data", 3) + numbered_names("usr_wf_F1_data", 3)


def new_numbered_names(prefix, count):
    return [f"{prefix}_{i}" for i in range (1, count)]

new_names = new_numbered_names("raw", 4) + new_numbered_names("fft", 4)

rename_map = dict(zip(measurement_names, new_names))

results = {}

for cm_folder in sorted(base_dir.iterdir()):
    if not cm_folder.is_dir():
        continue
    if "23" in cm_folder.name:
        continue

    cm_folder_name = cm_folder.name
    print(f"Processing folder: {cm_folder_name}")
    results[cm_folder_name] = {}

    for position in position_names:
        position_folder = cm_folder / f"metingen {position}"
        position_key = position_folder.name

        if not position_folder.is_dir():
            print(f"  Missing folder: {position_folder}")
            continue

        results[cm_folder_name][position_key] = {}

        for measurement in measurement_names:
            file_path = position_folder / f"{measurement}.csv"

            if file_path.exists():
                data = np.loadtxt(file_path,skiprows = 12, delimiter = ",")
                key_name = rename_map.get(measurement, measurement)
                results[cm_folder_name][position_key][key_name] = data
            else:
                print(f"     Missing file: {file_path}")

In [ ]:
# COLLECT ALL FFT ARRAYS

fft_arrays = []

for cm_name, positions in results.items():
    for position, measurements in positions.items():
        for measurement_name, data in measurements.items():
            if measurement_name.startswith("fft"):
                fft_arrays.append(data)

print(f"Collected {len(fft_arrays)} FFT arrays")


# MASSES AND TENSIONS

gewicht_1 = 1.0468   # kg
gewicht_2 = 1.0436   # kg
g = 9.81             # m/s^2

position_names = [f"0-{i}" for i in range(1, 5)] + [f"{i}-4" for i in range(1, 5)]

# hefboomverhoudingen
distances = [4, 6, 8, 10]

position_values = {}

# eerste vier posities: alleen gewicht 1
for name, d in zip(position_names[:4], distances):
    position_values[name] = gewicht_1 * g * d

# laatste vier posities:
# gewicht 1 blijft op verhouding 10 staan
# gewicht 2 wordt toegevoegd met verhouding 4, 6, 8, 10
for name, d in zip(position_names[4:], distances):
    position_values[name] = gewicht_1 * g * 10 + gewicht_2 * g * d

print("\nTensions:")
for name, tension in position_values.items():
    print(name, tension)


# EXTRACT FREQUENCY AND FFT AMPLITUDE

frequencies = []
magnitudes = []

for data in fft_arrays:
    frequencies.extend(data[:, 0])
    magnitudes.extend(data[:, 3])

frequencies = np.array(frequencies)
magnitudes = np.array(magnitudes)

print("\nNumber of frequency points:", len(frequencies))
print("Number of amplitude points:", len(magnitudes))

In [ ]:
cm_name = "metingen 31.5 cm"
position = "metingen 0-1"

fft_data = results[cm_name][position]["fft_1"]
raw_data = results[cm_name][position]["raw_1"]

f = fft_data[:, 0]
A = fft_data[:, 3]

t = raw_data[:, 0]
V = raw_data[:, 1]

print("FFT shape:", fft_data.shape)
print("RAW shape:", raw_data.shape)

print(raw_data[:5])





In [ ]:
# FFT calculated from the raw signal

# Remove DC offset
V_centered = V - np.mean(V)

# Number of samples
N = len(V_centered)

# Time step and sampling frequency
dt = np.mean(np.diff(t))
fs = 1 / dt

# Apply Hanning window
window = np.hanning(N)
V_windowed = V_centered * window

# Calculate FFT
fft_result = np.fft.rfft(V_windowed)
freqs = np.fft.rfftfreq(N, d=dt)

# Calculate amplitude
magnitude = np.abs(fft_result) / N
magnitude[1:-1] *= 2

# Convert to dB relative to maximum
magnitude_db = 20 * np.log10(
    np.maximum(magnitude, np.finfo(float).tiny)
    / np.max(magnitude)
)

# Only show 0–250 Hz
mask = freqs <= 250

plt.figure()
plt.plot(freqs[mask], magnitude_db[mask], linewidth=0.5)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Relative amplitude (dB)")
plt.title("FFT calculated from raw signal")
plt.grid()

plt.show()

plt.figure()
plt.plot(t, V)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Raw oscillation signal")
plt.savefig("oscillations_experiment.pdf", bbox_inches="tight")
plt.show()
plt.figure()
plt.plot(f, A, linewidth=0.5)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (dBV)")
plt.title("Oscilloscope FFT")

plt.xlim(0, 250)
plt.grid()

plt.show()

In [ ]:
from scipy.signal import find_peaks

# remove offset
V_centered = V - np.mean(V)

dt = np.mean(np.diff(t))

# dominante frequentie uit FFT
dominant_frequency = f[np.argmax(A)]

# aantal samples per periode
samples_per_period = 1 / (dominant_frequency * dt)

print("Dominant frequency:", dominant_frequency, "Hz")
print("Samples per period:", samples_per_period)
# find peaks in absolute signal
peaks, properties = find_peaks(
    V_filtered,
    distance=9,
    prominence=0.006
)

peak_times = t[peaks]
peak_amplitudes = V_filtered[peaks]

plt.figure(figsize=(10,5))

plt.plot(t, V_centered, linewidth=0.7, label="signal")
plt.plot(
    peak_times,
    peak_amplitudes,
    "o",
    markersize=3,
    label="detected peaks"
)

plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Oscillation with detected amplitudes")
plt.legend()
plt.grid()

plt.show()

coeff, cov = np.polyfit(
    peak_times,
    np.log(peak_amplitudes),
    1,
    cov=True
)

alpha = -coeff[0]
A0 = np.exp(coeff[1])
alpha_uncertainty = np.sqrt(cov[0, 0])

print("Damping factor =", alpha, "+/-", alpha_uncertainty, "1/s")

print("Damping factor =", alpha, "1/s")
print("Initial amplitude =", A0, "V")

A_fit = A0 * np.exp(-alpha * t)

plt.figure(figsize=(10,5))

plt.plot(t, V_centered, linewidth=0.7)
plt.plot(t, A_fit, "--", label="fitted envelope")
plt.plot(t, -A_fit, "--")

plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.legend()
plt.grid()
plt.show()
# lokale dempingsfactor tussen opeenvolgende pieken

alpha_local = np.log(
    peak_amplitudes[:-1] / peak_amplitudes[1:]
) / (
    peak_times[1:] - peak_times[:-1]
)

# tijdstip tussen beide pieken nemen als x-waarde
alpha_times = 0.5 * (
    peak_times[:-1] + peak_times[1:]
)

plt.figure(figsize=(10,5))

plt.plot(alpha_times, alpha_local, "o-", markersize=3)

plt.xlabel("Time (s)")
plt.ylabel("Local damping factor α (1/s)")
plt.title("Local damping factor as a function of time")
plt.grid()

plt.show()

window = 20

alpha_smooth = np.convolve(
    alpha_local,
    np.ones(window) / window,
    mode="valid"
)

time_smooth = alpha_times[window-1:]

plt.figure(figsize=(10,5))

plt.plot(alpha_times, alpha_local, alpha=0.25, label="Local α")
plt.plot(time_smooth, alpha_smooth, linewidth=2, label="Moving average")

plt.xlabel("Time (s)")
plt.ylabel("Damping factor α (1/s)")
plt.title("Damping factor as a function of time")
plt.legend()
plt.grid()

plt.show()
print("Mean smoothed alpha =", np.mean(alpha_smooth))
print("Median smoothed alpha =", np.median(alpha_smooth))
print("Min smoothed alpha =", np.min(alpha_smooth))
print("Max smoothed alpha =", np.max(alpha_smooth))

In [ ]:
from scipy.optimize import curve_fit

# Use only the part after the initial transient
mask = peak_times >= 0.3

peak_times_fit = peak_times[mask]
peak_amplitudes_fit = peak_amplitudes[mask]


# -------------------------
# Define the two models
# -------------------------

def exp_model(t, A0, beta):
    return A0 * np.exp(-beta * t)


def quad_model(t, A0, k):
    return A0 / (1 + k * A0 * t)


# -------------------------
# Fit exponential model
# -------------------------

popt_exp, _ = curve_fit(
    exp_model,
    peak_times_fit,
    peak_amplitudes_fit,
    p0=[peak_amplitudes_fit[0], 0.5]
)

A0_exp, beta_exp = popt_exp


# -------------------------
# Fit quadratic damping model
# -------------------------

popt_quad, _ = curve_fit(
    quad_model,
    peak_times_fit,
    peak_amplitudes_fit,
    p0=[peak_amplitudes_fit[0], 10]
)

A0_quad, k_quad = popt_quad


# -------------------------
# Predicted amplitudes
# -------------------------

A_exp_fit = exp_model(
    peak_times_fit,
    A0_exp,
    beta_exp
)

A_quad_fit = quad_model(
    peak_times_fit,
    A0_quad,
    k_quad
)


# -------------------------
# Calculate R²
# -------------------------

ss_tot = np.sum(
    (peak_amplitudes_fit - np.mean(peak_amplitudes_fit))**2
)

ss_res_exp = np.sum(
    (peak_amplitudes_fit - A_exp_fit)**2
)

R2_exp = 1 - ss_res_exp / ss_tot


ss_res_quad = np.sum(
    (peak_amplitudes_fit - A_quad_fit)**2
)

R2_quad = 1 - ss_res_quad / ss_tot


# -------------------------
# Print results
# -------------------------

print("Exponential fit")
print("A0 =", A0_exp)
print("beta =", beta_exp)
print("R² =", R2_exp)

print()

print("Quadratic damping fit")
print("A0 =", A0_quad)
print("k =", k_quad)
print("R² =", R2_quad)


# -------------------------
# Plot
# -------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    peak_times_fit,
    peak_amplitudes_fit,
    "o",
    markersize=3,
    label="Measured peaks"
)

plt.plot(
    peak_times_fit,
    A_exp_fit,
    linewidth=2,
    label="Exponential fit"
)

plt.plot(
    peak_times_fit,
    A_quad_fit,
    linewidth=2,
    label="Quadratic damping fit"
)

plt.xlabel("Time (s)")
plt.ylabel("Peak amplitude (V)")
plt.title("Comparison of damping models after initial transient")

plt.legend()
plt.grid()

plt.show()

In [ ]:
from scipy.optimize import curve_fit

model_results = []

# models
def exp_model(t, A0, beta):
    return A0 * np.exp(-beta * t)

def quad_model(t, A0, k):
    return A0 / (1 + k * A0 * t)


for cm_name, positions in results.items():
    for position_name, measurements in positions.items():

        for measurement_name, raw_data in measurements.items():

            if not measurement_name.startswith("raw"):
                continue

            # skip the one measurement we already identified as invalid
            if (
                cm_name == "metingen 25.5 cm"
                and position_name == "metingen 1-4"
                and measurement_name == "raw_2"
            ):
                continue

            try:
                # -------------------------
                # Raw data
                # -------------------------
                t = raw_data[:, 0]
                V = raw_data[:, 1]

                V_centered = V - np.mean(V)

                dt = np.mean(np.diff(t))
                fs = 1 / dt

                # -------------------------
                # Find dominant frequency
                # -------------------------
                N = len(V_centered)

                freqs = np.fft.rfftfreq(N, d=dt)
                fft_result = np.abs(np.fft.rfft(V_centered))

                # only search for the fundamental in the relevant range
                freq_mask = (freqs > 100) & (freqs < 600)

                dominant_frequency = freqs[freq_mask][
                    np.argmax(fft_result[freq_mask])
                ]

                # -------------------------
                # Band-pass filter
                # -------------------------
                lowcut = dominant_frequency - 8
                highcut = dominant_frequency + 8

                b, a = butter(
                    4,
                    [lowcut, highcut],
                    btype="bandpass",
                    fs=fs
                )

                V_filtered = filtfilt(b, a, V_centered)

                # -------------------------
                # Peak detection
                # -------------------------
                samples_per_period = fs / dominant_frequency
                distance = int(0.8 * samples_per_period)

                peaks, _ = find_peaks(
                    V_filtered,
                    distance=distance,
                    prominence=0.006
                )

                peak_times = t[peaks]
                peak_amplitudes = V_filtered[peaks]

                # -------------------------
                # Remove initial transient
                # -------------------------
                mask = peak_times >= 0.3

                peak_times_fit = peak_times[mask]
                peak_amplitudes_fit = peak_amplitudes[mask]

                # need enough points to fit
                if len(peak_times_fit) < 10:
                    print(
                        "Too few peaks:",
                        cm_name,
                        position_name,
                        measurement_name
                    )
                    continue

                # -------------------------
                # Exponential fit
                # -------------------------
                popt_exp, _ = curve_fit(
                    exp_model,
                    peak_times_fit,
                    peak_amplitudes_fit,
                    p0=[peak_amplitudes_fit[0], 0.5],
                    maxfev=10000
                )

                A0_exp, beta_exp = popt_exp

                A_exp_fit = exp_model(
                    peak_times_fit,
                    A0_exp,
                    beta_exp
                )

                # -------------------------
                # Quadratic damping fit
                # -------------------------
                popt_quad, _ = curve_fit(
                    quad_model,
                    peak_times_fit,
                    peak_amplitudes_fit,
                    p0=[peak_amplitudes_fit[0], 10],
                    maxfev=10000
                )

                A0_quad, k_quad = popt_quad

                A_quad_fit = quad_model(
                    peak_times_fit,
                    A0_quad,
                    k_quad
                )

                # -------------------------
                # R² values
                # -------------------------
                ss_tot = np.sum(
                    (
                        peak_amplitudes_fit
                        - np.mean(peak_amplitudes_fit)
                    )**2
                )

                ss_res_exp = np.sum(
                    (peak_amplitudes_fit - A_exp_fit)**2
                )

                ss_res_quad = np.sum(
                    (peak_amplitudes_fit - A_quad_fit)**2
                )

                R2_exp = 1 - ss_res_exp / ss_tot
                R2_quad = 1 - ss_res_quad / ss_tot

                # -------------------------
                # Store result
                # -------------------------
                position_short = position_name.replace(
                    "metingen ",
                    ""
                )

                model_results.append([
                    cm_name,
                    position_short,
                    measurement_name,
                    position_values[position_short],
                    dominant_frequency,
                    beta_exp,
                    k_quad,
                    R2_exp,
                    R2_quad
                ])

            except Exception as e:
                print(
                    "Problem with:",
                    cm_name,
                    position_name,
                    measurement_name,
                    e
                )


# -------------------------
# Put everything in a table
# -------------------------

df_models = pd.DataFrame(
    model_results,
    columns=[
        "wire_length",
        "position",
        "measurement",
        "tension_N",
        "frequency_Hz",
        "beta_exp",
        "k_quad",
        "R2_exp",
        "R2_quad"
    ]
)

df_models["best_model"] = np.where(
    df_models["R2_quad"] > df_models["R2_exp"],
    "quadratic",
    "exponential"
)

print("Number of analysed measurements:", len(df_models))

display(df_models)

In [ ]:
# Count which model fits better
model_counts = df_models["best_model"].value_counts()

print("Best fitting model:")
print(model_counts)

print()

# Percentages
model_percentages = (
    df_models["best_model"].value_counts(normalize=True) * 100
)

print("Percentages:")
print(model_percentages)

print()

# Mean R² for both models
print("Mean R² exponential:", df_models["R2_exp"].mean())
print("Mean R² quadratic:", df_models["R2_quad"].mean())

# Mean difference in R²
df_models["delta_R2"] = df_models["R2_quad"] - df_models["R2_exp"]

print()
print("Mean ΔR² (quadratic - exponential):",
      df_models["delta_R2"].mean())

In [ ]:
df_quad_grouped = (
    df_models
    .groupby(
        ["wire_length", "position", "tension_N"],
        as_index=False
    )
    .agg(
        n=("k_quad", "count"),
        k_mean=("k_quad", "mean"),
        k_std=("k_quad", "std"),
        R2_quad_mean=("R2_quad", "mean")
    )
)

display(df_quad_grouped)

In [ ]:
# Convert wire length names to length in metres
length_map = {
    "metingen 25.5 cm": 0.255,
    "metingen 27 cm":   0.270,
    "metingen 28.5 cm": 0.285,
    "metingen 30 cm":   0.300,
    "metingen 31.5 cm": 0.315
}

df_quad_grouped["L_m"] = (
    df_quad_grouped["wire_length"].map(length_map)
)

# Constants theoretical model
C_w = 1.2
rho = 1.204
D = 0.002
mu = 0.015
n = 1

# Theoretical damping coefficient
df_quad_grouped["k_theory"] = (
    (2 * C_w * rho * D * n)
    / (3 * mu * df_quad_grouped["L_m"]**2)
    * np.sqrt(df_quad_grouped["tension_N"] / mu)
)

# Ratio experiment / theory
df_quad_grouped["k_ratio"] = (
    df_quad_grouped["k_mean"]
    / df_quad_grouped["k_theory"]
)

display(
    df_quad_grouped[
        [
            "wire_length",
            "position",
            "tension_N",
            "k_mean",
            "k_std",
            "k_theory",
            "k_ratio"
        ]
    ]
)

In [ ]:
plt.figure(figsize=(10,6))

for length, group in df_quad_grouped.groupby("wire_length"):

    # experimental values
    plt.errorbar(
        group["tension_N"],
        group["k_mean"],
        yerr=group["k_std"],
        marker="o",
        capsize=4,
        label=f"{length} experiment"
    )

    # theoretical prediction
    plt.plot(
        group["tension_N"],
        group["k_theory"],
        "--",
        label=f"{length} theory"
    )

plt.xlabel("Tension T (N)")
plt.ylabel("Damping coefficient")
plt.title("Experimental and theoretical damping coefficient")
plt.grid()
plt.legend()
plt.savefig("Theorie versus experiment", bbox_inches="tight")
plt.show()


In [ ]:
# The theoretical model predicts:
# k ∝ sqrt(T) / L^2

df_quad_grouped["theory_scaling"] = (
    np.sqrt(df_quad_grouped["tension_N"])
    / df_quad_grouped["L_m"]**2
)

# Linear fit:
# k_exp = a * sqrt(T)/L^2 + b

coeff = np.polyfit(
    df_quad_grouped["theory_scaling"],
    df_quad_grouped["k_mean"],
    1
)

slope = coeff[0]
intercept = coeff[1]

k_fit = (
    slope * df_quad_grouped["theory_scaling"]
    + intercept
)

# R²
ss_res = np.sum(
    (df_quad_grouped["k_mean"] - k_fit)**2
)

ss_tot = np.sum(
    (
        df_quad_grouped["k_mean"]
        - df_quad_grouped["k_mean"].mean()
    )**2
)

R2_scaling = 1 - ss_res / ss_tot

print("Slope =", slope)
print("Intercept =", intercept)
print("R² theoretical scaling =", R2_scaling)


# Plot
plt.figure(figsize=(8,6))

plt.errorbar(
    df_quad_grouped["theory_scaling"],
    df_quad_grouped["k_mean"],
    yerr=df_quad_grouped["k_std"],
    fmt="o",
    capsize=3,
    label="Experiment"
)

x_line = np.linspace(
    df_quad_grouped["theory_scaling"].min(),
    df_quad_grouped["theory_scaling"].max(),
    200
)

plt.plot(
    x_line,
    slope*x_line + intercept,
    label="Linear fit"
)

plt.xlabel(r"$\sqrt{T}/L^2$")
plt.ylabel(r"Experimental damping coefficient $k$")
plt.title("Test of theoretical scaling")

plt.grid()
plt.legend()
plt.show()

In [ ]:
window = 20

alpha_window = []
time_window = []

for i in range(len(peak_times) - window + 1):
    t_part = peak_times[i:i+window]
    A_part = peak_amplitudes[i:i+window]

    coeff = np.polyfit(
        t_part,
        np.log(A_part),
        1
    )

    alpha_i = -coeff[0]

    alpha_window.append(alpha_i)
    time_window.append(np.mean(t_part))

alpha_window = np.array(alpha_window)
time_window = np.array(time_window)

plt.figure(figsize=(10,5))

plt.plot(time_window, alpha_window)

plt.axhline(
    alpha,
    linestyle="--",
    label="global alpha"
)

plt.xlabel("Time (s)")
plt.ylabel("Damping factor α (1/s)")
plt.title("Local damping factor from sliding exponential fit")
plt.legend()
plt.grid()

plt.show()
plt.figure(figsize=(10,5))

plt.plot(peak_times, peak_amplitudes, "o-", markersize=3)

plt.xlabel("Time (s)")
plt.ylabel("Peak amplitude (V)")
plt.title("Peak amplitude as a function of time")
plt.grid()

plt.show()

In [ ]:
def analyse_damping(raw_data):
    t = raw_data[:, 0]
    V = raw_data[:, 1]

    # remove offset
    V_centered = V - np.mean(V)

    # sampling frequency
    dt = np.mean(np.diff(t))
    fs = 1 / dt

    # find dominant frequency from FFT of raw data
    N = len(V_centered)
    freqs = np.fft.rfftfreq(N, d=dt)
    fft_result = np.abs(np.fft.rfft(V_centered))

    # ignore 0 Hz
    freq_mask = (freqs > 100) & (freqs < 600)

    dominant_frequency = freqs[freq_mask][
    np.argmax(fft_result[freq_mask])
    ]
    # band-pass filter around dominant frequency
    lowcut = dominant_frequency - 8
    highcut = dominant_frequency + 8

    b, a = butter(
        4,
        [lowcut, highcut],
        btype="bandpass",
        fs=fs
    )

    V_filtered = filtfilt(b, a, V_centered)

    # approximately one positive peak per period
    samples_per_period = fs / dominant_frequency
    distance = int(0.8 * samples_per_period)

    # detect positive peaks
    peaks, _ = find_peaks(
        V_filtered,
        distance=distance,
        prominence=0.006
    )

    peak_times = t[peaks]
    peak_amplitudes = V_filtered[peaks]

    # exponential fit
    coeff, cov = np.polyfit(
        peak_times,
        np.log(peak_amplitudes),
        1,
        cov=True
    )

    alpha = -coeff[0]
    A0 = np.exp(coeff[1])

    alpha_uncertainty = np.sqrt(cov[0, 0])
    A0_uncertainty = A0 * np.sqrt(cov[1, 1])

    return (
        alpha,
        alpha_uncertainty,
        A0,
        A0_uncertainty,
        dominant_frequency
    )

result = analyse_damping(raw_data)

print(result)

In [ ]:
all_results = []

for cm_name, positions in results.items():
    for position_name, measurements in positions.items():
        for measurement_name, data in measurements.items():

            if measurement_name.startswith("raw"):
                try:
                    alpha, alpha_unc, A0, A0_unc, f_dom = analyse_damping(data)

                    # remove "metingen " from the position name
                    position_short = position_name.replace("metingen ", "")

                    tension = position_values[position_short]

                    all_results.append([
                        cm_name,
                        position_short,
                        measurement_name,
                        tension,
                        f_dom,
                        alpha,
                        alpha_unc,
                        A0,
                        A0_unc
                    ])

                except Exception as e:
                    print(
                        "Problem with:",
                        cm_name,
                        position_name,
                        measurement_name,
                        e
                    )

print("Number of analysed measurements:", len(all_results))

In [ ]:
import pandas as pd

df_results = pd.DataFrame(
    all_results,
    columns=[
        "wire_length",
        "position",
        "measurement",
        "tension_N",
        "frequency_Hz",
        "alpha_1_s",
        "alpha_unc_1_s",
        "A0_V",
        "A0_unc_V"
    ]
)

display(df_results)

In [ ]:
df_grouped = (
    df_results
    .groupby(["wire_length", "position", "tension_N"])
    .agg(
        frequency_mean=("frequency_Hz", "mean"),
        frequency_std=("frequency_Hz", "std"),

        alpha_mean=("alpha_1_s", "mean"),
        alpha_std=("alpha_1_s", "std"),

        A0_mean=("A0_V", "mean"),
        A0_std=("A0_V", "std")
    )
    .reset_index()
)

display(df_grouped)

In [ ]:
plt.figure(figsize=(10,6))

for wire_length, group in df_grouped.groupby("wire_length"):

    group = group.sort_values("tension_N")

    plt.errorbar(
        group["tension_N"],
        group["alpha_mean"],
        yerr=group["alpha_std"],
        marker="o",
        capsize=4,
        label=wire_length
    )

plt.xlabel("Tension T (N)")
plt.ylabel("Damping factor α (1/s)")
plt.title("Damping factor as a function of tension")

plt.legend()
plt.grid()

plt.show()

In [ ]:
problem_points = df_results[
    (
        (df_results["wire_length"] == "metingen 25.5 cm") &
        (df_results["position"] == "1-4")
    )
    |
    (
        (df_results["wire_length"] == "metingen 30 cm") &
        (df_results["position"].isin(["1-4", "3-4"]))
    )
]

display(
    problem_points[
        [
            "wire_length",
            "position",
            "measurement",
            "tension_N",
            "frequency_Hz",
            "alpha_1_s",
            "alpha_unc_1_s",
            "A0_V"
        ]
    ]
)

In [ ]:
test_data = results["metingen 31.5 cm"]["metingen 0-1"]["raw_1"]
print(analyse_damping(test_data))
t_test = test_data[:, 0]
V_test = test_data[:, 1]

V_test_centered = V_test - np.mean(V_test)

dt_test = np.mean(np.diff(t_test))
fs_test = 1 / dt_test

# FFT
N = len(V_test_centered)
freqs_test = np.fft.rfftfreq(N, d=dt_test)
fft_test = np.abs(np.fft.rfft(V_test_centered))

freq_mask = (freqs_test > 100) & (freqs_test < 600)

f_dom_test = freqs_test[freq_mask][
    np.argmax(fft_test[freq_mask])
]

# band-pass around dominant frequency
b, a = butter(
    4,
    [f_dom_test - 8, f_dom_test + 8],
    btype="bandpass",
    fs=fs_test
)

V_test_filtered = filtfilt(b, a, V_test_centered)

samples_per_period = fs_test / f_dom_test
distance = int(0.8 * samples_per_period)

peaks_test, _ = find_peaks(
    V_test_filtered,
    distance=distance,
    prominence=0.006
)

peak_times_test = t_test[peaks_test]
peak_amplitudes_test = V_test_filtered[peaks_test]

plt.figure(figsize=(10,5))

plt.plot(t_test, V_test_centered, alpha=0.35, label="Original")
plt.plot(t_test, V_test_filtered, label="Filtered")
plt.plot(
    peak_times_test,
    peak_amplitudes_test,
    "o",
    markersize=3,
    label="Detected peaks"
)

plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("25.5 cm – 143.6 N – raw_2")
plt.legend()
plt.grid()
plt.show()

In [ ]:
outlier_check = []

for (wire_length, position), group in df_results.groupby(
    ["wire_length", "position"]
):
    if len(group) != 3:
        continue

    alpha_values = group["alpha_1_s"].values

    mean_alpha = np.mean(alpha_values)
    std_alpha = np.std(alpha_values, ddof=1)

    relative_std = std_alpha / mean_alpha

    outlier_check.append([
        wire_length,
        position,
        mean_alpha,
        std_alpha,
        relative_std
    ])

df_outliers = pd.DataFrame(
    outlier_check,
    columns=[
        "wire_length",
        "position",
        "alpha_mean",
        "alpha_std",
        "relative_std"
    ]
)

df_outliers = df_outliers.sort_values(
    "relative_std",
    ascending=False
)

display(df_outliers)

In [ ]:
df_clean = df_results[
    ~(
        (df_results["wire_length"] == "metingen 25.5 cm") &
        (df_results["position"] == "1-4") &
        (df_results["measurement"] == "raw_2")
    )
].copy()

print("Original measurements:", len(df_results))
print("Measurements used:", len(df_clean))

In [ ]:
df_grouped_clean = (
    df_clean
    .groupby(["wire_length", "position", "tension_N"])
    .agg(
        n=("alpha_1_s", "count"),

        frequency_mean=("frequency_Hz", "mean"),
        frequency_std=("frequency_Hz", "std"),

        alpha_mean=("alpha_1_s", "mean"),
        alpha_std=("alpha_1_s", "std"),

        A0_mean=("A0_V", "mean"),
        A0_std=("A0_V", "std")
    )
    .reset_index()
)

display(df_grouped_clean)

In [ ]:
df_grouped_clean["alpha_mean_unc"] = np.sqrt(
    (df_grouped_clean["alpha_std"] / np.sqrt(df_grouped_clean["n"]))**2
    +
    (df_grouped_clean["alpha_fit_unc"] / np.sqrt(df_grouped_clean["n"]))**2
)

plt.figure(figsize=(10,6))

for wire_length, group in df_grouped_clean.groupby("wire_length"):

    group = group.sort_values("tension_N")

    plt.errorbar(
        group["tension_N"],
        group["alpha_mean"],
        yerr=group["alpha_mean_unc"],
        marker="o",
        capsize=4,
        label=wire_length
    )

plt.xlabel("Tension T (N)")
plt.ylabel("Damping factor α (1/s)")
plt.title("Damping factor as a function of tension")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# Mean fit uncertainty in A0 for each condition
A0_fit_unc = (
    df_clean
    .groupby(["wire_length", "position", "tension_N"])
    ["A0_unc_V"]
    .apply(lambda x: np.sqrt(np.mean(x**2)))
    .reset_index(name="A0_fit_unc")
)

# Add to grouped dataframe
df_grouped_clean = df_grouped_clean.merge(
    A0_fit_unc,
    on=["wire_length", "position", "tension_N"]
)

# Uncertainty of the mean A0
df_grouped_clean["A0_mean_unc"] = np.sqrt(
    (df_grouped_clean["A0_std"] / np.sqrt(df_grouped_clean["n"]))**2
    +
    (df_grouped_clean["A0_fit_unc"] / np.sqrt(df_grouped_clean["n"]))**2
)

In [ ]:
df_grouped_clean["tension_unc_N"] = df_grouped_clean["position"].map(
    position_tension_unc
)

plt.figure(figsize=(10,6))

for wire_length, group in df_grouped_clean.groupby("wire_length"):

    group = group.sort_values("tension_N")

    plt.errorbar(
        group["tension_N"],
        group["alpha_mean"],
        xerr=group["tension_unc_N"],
        yerr=group["alpha_mean_unc"],
        marker="o",
        capsize=4,
        label=wire_length
)

plt.xlabel("Tension T (N)")
plt.ylabel("Initial amplitude A0 (V)")
plt.title("Damping factor as a function of tension")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# measurement uncertainties
sigma_m1 = 0.0001   # kg
sigma_m2 = 0.0001   # kg

sigma_weight_pos = 0.001   # 0.1 cm = 0.001 m
sigma_other_arm = 0.001    # 0.1 cm = 0.001 m

m1 = 1.0468
m2 = 1.0436
g = 9.81

# lever arms in meters
weight_positions = [0.08, 0.12, 0.16, 0.20]
other_arm = 0.02

In [ ]:
# tension uncertainties for all positions

position_tension_unc = {}

# first 4 positions: only m1 contributes
for name, x in zip(position_names[:4], weight_positions):

    T = m1 * g * x / other_arm

    rel_unc = np.sqrt(
        (sigma_m1 / m1)**2
        + (sigma_weight_pos / x)**2
        + (sigma_other_arm / other_arm)**2
    )

    position_tension_unc[name] = T * rel_unc


# last 4 positions: m1 at 20 cm + m2 at x
for name, x in zip(position_names[4:], weight_positions):

    T1 = m1 * g * 0.20 / other_arm
    T2 = m2 * g * x / other_arm

    # uncertainty in each contribution
    sigma_T1 = T1 * np.sqrt(
        (sigma_m1 / m1)**2
        + (sigma_weight_pos / 0.20)**2
        + (sigma_other_arm / other_arm)**2
    )

    sigma_T2 = T2 * np.sqrt(
        (sigma_m2 / m2)**2
        + (sigma_weight_pos / x)**2
        + (sigma_other_arm / other_arm)**2
    )

    # independent contributions added quadratically
    position_tension_unc[name] = np.sqrt(
        sigma_T1**2 + sigma_T2**2
    )


print(position_tension_unc)

In [ ]:
# Mean fit uncertainty for each measurement condition
fit_unc = (
    df_clean
    .groupby(["wire_length", "position", "tension_N"])
    ["alpha_unc_1_s"]
    .apply(lambda x: np.sqrt(np.mean(x**2)))
    .reset_index(name="alpha_fit_unc")
)

# Add to grouped dataframe
df_grouped_clean = df_grouped_clean.merge(
    fit_unc,
    on=["wire_length", "position", "tension_N"]
)

# Combine repeatability and fit uncertainty
df_grouped_clean["alpha_total_unc"] = np.sqrt(
    df_grouped_clean["alpha_std"]**2 +
    df_grouped_clean["alpha_fit_unc"]**2
)

display(
    df_grouped_clean[
        [
            "wire_length",
            "position",
            "alpha_mean",
            "alpha_std",
            "alpha_fit_unc",
            "alpha_total_unc"
        ]
    ]
)

In [ ]:
df_grouped_clean["t_1pct_exp"] = (
    np.log(100) / df_grouped_clean["alpha_mean"]
)

display(
    df_grouped_clean[
        ["wire_length", "position", "tension_N", "t_1pct_exp"]
    ]
)

In [ ]:
length_map = {
    "metingen 25.5 cm": 0.255,
    "metingen 27 cm":   0.270,
    "metingen 28.5 cm": 0.285,
    "metingen 30 cm":   0.300,
    "metingen 31.5 cm": 0.315
}

df_grouped_clean["L_m"] = df_grouped_clean["wire_length"].map(length_map)

In [ ]:
C_w = 1.2
rho = 1.204
D = 0.002
mu = 0.015
n = 1



df_grouped_clean["alpha_theory"] = (
    (2 * C_w * rho * D * n)
    / (3 * mu * df_grouped_clean["L_m"]**2)
    * np.sqrt(df_grouped_clean["tension_N"] / mu)
)
# Theoretical time until amplitude has decreased to 1%
# U(t) = U0 / (1 + alpha_theory * U0 * t)

U0_theory = 0.01

df_grouped_clean["t_1pct_theory"] = (
    99 / (df_grouped_clean["alpha_theory"] * U0_theory)
)
df_grouped_clean["t_1pct_exp"] = (
    np.log(100) / df_grouped_clean["alpha_mean"]
)
display(
    df_grouped_clean[
        [
            "wire_length",
            "position",
            "tension_N",
            "t_1pct_exp",
            "t_1pct_theory"
        ]
    ]
)